In [1]:
import torch
print(torch.__version__)
dataset_path = "../data/raw"

2.12.0+cpu


In [2]:
x = torch.tensor([1,2,3,4])
print(x)
print(type(x))

tensor([1, 2, 3, 4])
<class 'torch.Tensor'>


In [3]:
y = torch.tensor([
    [1,2,3],
    [4,5,6]
])
print(y.shape)
print(y[0])
print(y[1])
print(y[0][0])
print(x.shape)

torch.Size([2, 3])
tensor([1, 2, 3])
tensor([4, 5, 6])
tensor(1)
torch.Size([4])


In [4]:
from torchvision.datasets import ImageFolder
dataset = ImageFolder(dataset_path)

In [5]:
print(len(dataset))
print(dataset.class_to_idx)

20638
{'Pepper__bell___Bacterial_spot': 0, 'Pepper__bell___healthy': 1, 'Potato___Early_blight': 2, 'Potato___Late_blight': 3, 'Potato___healthy': 4, 'Tomato_Bacterial_spot': 5, 'Tomato_Early_blight': 6, 'Tomato_Late_blight': 7, 'Tomato_Leaf_Mold': 8, 'Tomato_Septoria_leaf_spot': 9, 'Tomato_Spider_mites_Two_spotted_spider_mite': 10, 'Tomato__Target_Spot': 11, 'Tomato__Tomato_YellowLeaf__Curl_Virus': 12, 'Tomato__Tomato_mosaic_virus': 13, 'Tomato_healthy': 14}


In [6]:
img,label = dataset[0]
print(type(img))
print(label)

<class 'PIL.Image.Image'>
0


In [7]:
print(dataset.classes[label])

Pepper__bell___Bacterial_spot


In [8]:
from torchvision import transforms
transform  = transforms.ToTensor()
dataset = ImageFolder(
    dataset_path,transform= transform
)
img,label = dataset[0]
print(type(img))
print(img.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([3, 256, 256])
0


In [9]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
0
No GPU


In [10]:
from torch.utils.data import DataLoader
loader = DataLoader(
    dataset,
    batch_size= 32,
    shuffle= True
)
images,labels = next(iter(loader))
print(images.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 3, 256, 256])
torch.Size([32])
tensor([ 9, 12, 12,  7,  8,  0, 14,  3,  7,  5])


In [11]:
import torch.nn as nn
layer = nn.Linear(5,3)
x = torch.tensor(
    [[1,2,3,4,5]],
    dtype = torch.float32
)
output = layer(x)

print(x.shape)
print(output.shape)
print(output)

torch.Size([1, 5])
torch.Size([1, 3])
tensor([[ 1.7530, -3.0340,  2.2767]], grad_fn=<AddmmBackward0>)


In [12]:
print(nn)

<module 'torch.nn' from 'c:\\Users\\shukl\\OneDrive\\Desktop\\crop-disease-detector\\.venv\\Lib\\site-packages\\torch\\nn\\__init__.py'>


In [13]:
conv = nn.Conv2d(
    in_channels = 3,
    out_channels=  32,
    kernel_size= 3
) 
x = torch.randn(32,3,256,256)
output = conv(x);
print(x.shape)
print(output.shape)

torch.Size([32, 3, 256, 256])
torch.Size([32, 32, 254, 254])


In [14]:
relu = nn.ReLU();
y = torch.tensor([-5,-6,0,3,7])
print(relu(y))

tensor([0, 0, 0, 3, 7])


In [15]:
max = nn.MaxPool2d(kernel_size=2)
print(max(x).shape)

torch.Size([32, 3, 128, 128])


In [16]:
flat = nn.Flatten()
print(flat(x).shape)

torch.Size([32, 196608])


In [17]:

class CropDisease(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32,64,3)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(64,128,3)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2)

        self.flat = nn.Flatten()
        self.fc = nn.Linear(115200,15)
    
    def forward(self,x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.flat(x)
        x = self.fc(x)
        return x

In [18]:
model = CropDisease()
print(model)

CropDisease(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flat): Flatten(start_dim=1, end_dim=-1)
  (fc): Linear(in_features=115200, out_features=15, bias=True)
)


In [19]:
print(model(x).shape)

torch.Size([32, 15])


In [20]:
lossFunction = nn.CrossEntropyLoss()
print(lossFunction)

CrossEntropyLoss()


In [21]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [22]:
for images,labels in loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 256, 256])
torch.Size([32])


In [23]:
output = model(images)
loss = lossFunction(output,labels)
print(loss)

tensor(2.7100, grad_fn=<NllLossBackward0>)


In [24]:
optimizer.zero_grad
loss.backward()
optimizer.step()
print("done")

done


In [25]:
epochs = 5
for epoch in range(epochs):
    for images,labels in loader:
        output = model(images)
        loss = lossFunction(output,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("Epoch :",epoch+1,"Loss :",loss.item())

Epoch : 1 Loss : 0.45604991912841797
Epoch : 2 Loss : 0.30354034900665283
Epoch : 3 Loss : 0.17896904051303864
Epoch : 4 Loss : 0.15138207376003265
Epoch : 5 Loss : 0.015025225467979908


In [26]:
print(len(dataset))

20638


In [27]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)

2.12.0+cpu
CUDA available: False
Torch CUDA version: None


In [1]:
import torch

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())

Torch Version: 2.11.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Count: 1
